In [ ]:
import pandas as pd
from pyproj import Transformer

In [ ]:
PATH = r"c:\Users\marta\UNIR\HERRAMIENTAS_VISU\actividad3\stopsemt.csv" # sustituye aquí por la ruta donde tengas el archivo stopsemt.csv. El script lee de aquí y lo sobreescribe una vez procesado.
# cargar archivo
df = pd.read_csv(PATH,encoding="latin1", sep=",")
print(df)

# filtros y preprocesos iniciales
# filtrar solo sentido = 1
df = df[df["sentido"] == 1]

# renombrar columna line → Linea
df = df.rename(columns={"line": "Linea"})

# crear columna Nombre Linea
df["Nombre Linea"] = df["nameFrom"] + " - " + df["nameTo"]

# eliminar filas con nulos
df = df.dropna(subset=["posX", "posY"])

# eliminar valores  inválidos
df = df[(df["posX"] > 100000) & (df["posX"] < 10000000)]
df = df[(df["posY"] > 1000000) & (df["posY"] < 100000000)]

print("Datos limpiados")
print("Filas restantes:", len(df))

# configurar el transformador de EPSG:25830 a EPSG:4326
# 'always_xy=True' asegura que posX mapee a Longitud y posY a Latitud
transformer = Transformer.from_crs("EPSG:25830", "EPSG:4326", always_xy=True)

# aplicar la transformación en lote a tus columnas posX y posY
# .values convierte las columnas en listas eficientes para pyproj
longitudes, latitudes = transformer.transform(df["posX"].values, df["posY"].values)

# crear las nuevas columnas en tu DataFrame actual
df["longitud"] = longitudes
df["latitud"] = latitudes

# eliminar valores infinitos o NaN
df = df.replace([float("inf"), float("-inf")], pd.NA)
df = df.dropna(subset=["latitud", "longitud"])
print("Infinitos / NaN limpiados")
print("Filas restantes:", len(df))

# filtrar coordenadas para quedarnos solo con Madrid
df = df[
    (df["latitud"] > 39) & (df["latitud"] < 41) &
    (df["longitud"] > -5) & (df["longitud"] < -2)
]
print("Resultados filtrados por coordenadas de Madrid")
print("Filas restantes:", len(df))

# guardar el archivo final con todas las columnas originales + las nuevas
df.to_csv(PATH, index=False)
print("Conversión completada")

       line     dateIni              dateEnd label               nameTo  \
0         1  2026-06-13  2999-12-31 23:59:59     1  PLAZA DE CRISTO REY   
1         1  2026-06-13  2999-12-31 23:59:59     1  PLAZA DE CRISTO REY   
2         1  2026-06-13  2999-12-31 23:59:59     1  PLAZA DE CRISTO REY   
3         1  2026-06-13  2999-12-31 23:59:59     1  PLAZA DE CRISTO REY   
4         1  2026-06-13  2999-12-31 23:59:59     1  PLAZA DE CRISTO REY   
...     ...         ...                  ...   ...                  ...   
12697   898  2025-10-06  2999-12-31 23:59:59   SE2       PLAZA ELIPTICA   
12698   898  2025-10-06  2999-12-31 23:59:59   SE2       PLAZA ELIPTICA   
12699   898  2025-10-06  2999-12-31 23:59:59   SE2       PLAZA ELIPTICA   
12700   898  2025-10-06  2999-12-31 23:59:59   SE2       PLAZA ELIPTICA   
12701   898  2025-10-06  2999-12-31 23:59:59   SE2       PLAZA ELIPTICA   

             nameFrom  parada  secuencia  sentido  distancia  \
0         PROSPERIDAD    4514      

In [13]:
print(df[["latitud", "longitud"]].describe())


       latitud  longitud
count      0.0       0.0
mean       NaN       NaN
std        NaN       NaN
min        NaN       NaN
25%        NaN       NaN
50%        NaN       NaN
75%        NaN       NaN
max        NaN       NaN
